## POD Line Buffer Influence on Average Delta SDI

This workflow analyzes how SDI (Structure Defensibility Index) values change with distance from Potential Operational Delineation (POD) lines to identify the point where further buffering no longer adds meaningful variation.

First, the script processes POD lines by dissolving them into single-part features and creating buffered zones at multiple distances. It then runs zonal statistics on a set of Delta SDI raster files to calculate the mean SDI value within each buffer, compiling the results into a final table.

Next, the script analyzes how SDI values stabilize across buffer distances by computing the rate of change and applying a changepoint detection algorithm (ruptures). The detected stabilization distance is mapped back to the corresponding buffer size and visualized through graphs showing SDI trends and rate of change.

This workflow provides a structured approach to identifying the optimal buffer size

#### Import libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ruptures as rpt
import arcpy, os
from arcpy.sa import *

#### Calling in datasets Create variables
Import the necessary datasets for processing the Delta SDI data. This includes the file path to the POD lines shapefile or feature class, as well as the list of Delta SDI rasters that are of interest for the analysis. 

In [ ]:
#input the file path for the POD lines shapefile or feautre class for .gdb
InPOD = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\MBR\MBR_PODs\MBR_PODs.gdb\MBR_PODs_Line'

#define the file path and list of delta SDI rasters of interest for processing
delta_path = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\MBR\SDI\Delta_SDI' #file path to delt SDI rasters
delta_trts = ['mech_thin_delta_sdi_Pct90_MBR', 'patch_cut_sm_delta_sdi_Pct90_MBR','hand_thin_delta_sdi_Pct90_MBR', 'mast_delta_sdi_Pct90_MBR'] #list of delta SDI rasters of interest 
delta_files = [os.path.join(delta_path, f'{name}.tif') for name in delta_trts] #join file path and raster names to create a complete file path to each raster in list

#range of buffer distances user is intreseted in 
#(start=##, stop=####, interval=50)
buffer_distances = [f"{i} Feet" for i in range(50, 1525, 50)]


#this script generates multiple intermediate outputs
#set the option below to "Yes" or "No" to decide whether to delete them after execution
delete_scratch_gdb = 'Yes'  

#### Set environments

In [ ]:
from arcpy import env #setting the workspace and environment
workspace = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\MBR\SDI' #change this to match your directory structure this refers to the SDI folder created by the Delt SDI script

#set up scratch workspace 
#the necessary folders and .gdb file will be created in the folder structure setup section below
arcpy.env.workspace = workspace + r'\POD_Line_Delta_SDI\Scratch.gdb' 

#input spatial reference
#NAD83 UTM Zone12 = 26912|NAD83 UTM Zone13 = 26913|North American Albers Equal Area Conic = 102008
env.outputCoordinateSystem = arcpy.SpatialReference(26913) #set spatial reference 

env.overwriteOutput = True #sets outputs to overwrite

arcpy.env.addOutputsToMap = False #stops outputs from being displayed in TOC

#### Set up folder structure
Create folders to organize the data prepared by this script and maintain a standardized folder structure, which will facilitate integration of the Delta SDI script. This process also includes establishing a scratch and final .gdb designed for the intermediate data generated during the POD line segmentation analysis.

In [ ]:
#create the POD_Line_Buffer_On_SDI folder as the root directory for all output data data related to analysis
shared_output_folder = os.path.join(workspace, 'POD_Line_Buffer_Influence_Avg_Delta_SDI') 
if not os.path.exists(shared_output_folder):
    os.makedirs(shared_output_folder)
    
#create a file .gdb file for the scratch workspace to store intermediate data generated during the POD line segmentation analysis
scratch_gdb_path = os.path.join(shared_output_folder, 'Scratch.gdb')
if not os.path.exists(scratch_gdb_path):
    arcpy.CreateFileGDB_management(shared_output_folder, 'Scratch')
    
# #create a file .gdb to store the final outputs for deliverables 
# output_gdb = os.path.join(delta_sdi_lines_folder, 'POD_Line_Delta_SDI.gdb')
# if not os.path.exists(output_gdb):
#     arcpy.CreateFileGDB_management(delta_sdi_lines_folder, 'POD_Line_Delta_SDI')

####  Buffering POD Lines and Running Zonal Statistics

This code processes POD (Potential Operational Delineation) lines by applying a series of spatial analyses to measure how SDI (Structure Defensibility Index) values change across different buffer distances. The main objective is to calculate zonal statistics within buffered areas surrounding POD lines and compile the results into a final table for further analysis.

The script begins by dissolving POD lines into single-part features, ensuring each segment is treated separately. It then iterates through a list of buffer distances, creating buffered zones around the POD lines to define areas of influence at varying scales. For each buffered zone, the script runs Zonal Statistics As Table on a set of Delta SDI raster files, calculating the mean SDI value within each buffer. These results are then compiled into a final table.

#### Creating Graphs and Detecting the Leveling-Off Point
This code analyzes how mean SDI (Structure Defensibility Index) values change with increasing buffer distances to determine the point where further buffering no longer adds meaningful variation. The goal is to identify the stabilization distance, beyond which additional buffering primarily captures redundant information rather than new insights.

The script begins by extracting and formatting SDI data from a CSV file, converting buffer distances into a numerical format for analysis. It then calculates the mean SDI value at each buffer distance and measures how much SDI changes between successive buffers. Using the ruptures package, the code applies a changepoint detection algorithm to identify where the rate of SDI change stabilizes. Once a stabilization point is identified, it is mapped back to the corresponding buffer distance. The results are visualized through two plots: one showing the mean SDI trend across buffers, and another displaying the rate of SDI change, both highlighting where stabilization occurs.

##### Description of How ruptures Detects the Leveling-Off Point
The ruptures package is used to detect the buffer distance at which the mean SDI values stop changing significantly, meaning that further increases in buffer size do not add much new variation. This process is done using the Pruned Exact Linear Time (Pelt) algorithm with the Radial Basis Function (RBF) model, which is well-suited for detecting gradual shifts in data.

First, the script calculates the first difference of the mean SDI values across successive buffer distances. This represents the rate of change in SDI as the buffer expands. If the rate of change is large, it means that increasing the buffer distance is capturing additional variation. If the rate of change is small, it suggests that the SDI values are stabilizing.

The ruptures package is then applied to these difference values to identify the point at which the rate of change significantly decreases. The Pelt algorithm efficiently scans the data and segments it based on where meaningful shifts occur. The RBF model is used to compare the similarity of different segments, ensuring that it detects gradual stabilization rather than abrupt changes. A penalty parameter (pen=10) is used to control sensitivity—lower values detect stabilization earlier, while higher values require stronger evidence before marking a stabilization point.

Once a changepoint is detected, the script maps it back to the corresponding buffer distance. This represents the stabilization distance, or the point where additional buffering no longer provides meaningful new information about SDI variation. In the final output, this distance (e.g., 1550 feet) is identified as the point beyond which further increases in buffer size primarily capture redundant information rather than new variation.

In [ ]:
#dissolve pod lines into single-part features so that each segment is treated independently
print('Dissolving POD lines...')
POD_single_part = arcpy.management.Dissolve(InPOD, os.path.join(scratch_gdb_path, 'POD_Line_Dissolve'), 
                                            None, None, "SINGLE_PART", "DISSOLVE_LINES")  #ensures all lines are separate geometries

#loop through each delta SDI raster and perform analysis
for delta_file in delta_files:
    #get raster label by removing suffix
    label = os.path.basename(delta_file).split('delta_sdi')[0].rstrip('_')

    #create a dedicated output folder for this raster
    delta_sdi_lines_folder = os.path.join(shared_output_folder, label)
    if not os.path.exists(delta_sdi_lines_folder):
        os.makedirs(delta_sdi_lines_folder)

    #create output GDB specific to the raster
    output_gdb = os.path.join(delta_sdi_lines_folder, f'POD_Line_Delta_SDI_{label}.gdb')
    if not os.path.exists(output_gdb):
        arcpy.CreateFileGDB_management(delta_sdi_lines_folder, f'POD_Line_Delta_SDI_{label}')

    #initialize final zonal stats table path and tracking flag
    final_table = os.path.join(output_gdb, 'Final_ZonalStats_Table')
    print(f"[{label}] Created output GDB and initialized final table path.")
    #set a flag to track whether the final table needs to be initialized with the first valid dataset
    first_iteration = True

    #loop through each buffer distance to generate zonal statistics
    for buffer_distance in buffer_distances:
        print(f'[{label}] Processing buffer distance: {buffer_distance}')
        
        #create a buffer around the pod lines to represent the area of influence at the current distance
        POD_Buffer = arcpy.analysis.PairwiseBuffer(POD_single_part, 
                                               os.path.join(scratch_gdb_path, f'POD_Line_Buffer_{buffer_distance.replace(" ", "")}'), 
                                               buffer_distance)
        #set output path for zonal stats table
        output_table = os.path.join(scratch_gdb_path, f'ZonalStat_{label}_{buffer_distance.replace(" ", "")}')

        #run zonal statistics to calculate mean values of the raster within the buffered areas
        arcpy.sa.ZonalStatisticsAsTable(POD_Buffer, 'ORIG_FID', delta_file, output_table, 'DATA', 'MEAN')

        #check if the zonal statistics table has data to avoid joining empty results
        record_count = int(arcpy.GetCount_management(output_table)[0])
        if record_count == 0:
            print(f"[{label}] WARNING: No data for {buffer_distance}. Skipping.")
            continue
        
        #define a new mean field name based on the buffer distance
        new_mean_field = f"mean_{buffer_distance.replace(' ', '')}"

        #rename the 'MEAN' field in the zonal statistics output table for clarity
        arcpy.AlterField_management(output_table, 'MEAN', new_mean_field, new_mean_field)


        #if this is the first valid table, initialize the final output table with this data
        if first_iteration:
            arcpy.management.CopyRows(output_table, final_table)
            print(f"[{label}] Initialized final table with data from {buffer_distance}.")
            first_iteration = False
        else:
            #ensure that the final table has the required ORIG_FID field before joining
            #join the new zonal statistics results to the final table using ORIG_FID
            arcpy.management.JoinField(final_table, 'ORIG_FID', output_table, 'ORIG_FID', [new_mean_field])
            print(f"[{label}] Added field {new_mean_field} for buffer distance: {buffer_distance}")
    
    print('All buffer distance processing completed. Now exporting final table to CSV...')
    #export the final aggregated zonal statistics table to a csv file for external analysis
    csv_output_path = os.path.join(delta_sdi_lines_folder, f'Final_ZonalStats_{label}.csv')
    arcpy.conversion.TableToTable(final_table, delta_sdi_lines_folder, f'Final_ZonalStats_{label}.csv')
    print(f"[{label}] Exported CSV: {csv_output_path}")    

    #load the csv file containing sdi values at different buffer distances
    df = pd.read_csv(csv_output_path)  #read the file into a pandas dataframe
    
    #select only the columns that contain "mean_" since these represent buffer distances
    distance_cols = [col for col in df.columns if "mean_" in col]  #filter relevant columns
    
    #reshape the dataframe from wide to long format so that each buffer distance is a row
    df_long = df.melt(value_vars=distance_cols, var_name="Buffer_Distance", value_name="SDI_Value")
    
    #extract numeric values from buffer distance column names (e.g., "mean_50Feet" -> 50)
    df_long["Buffer_Distance"] = df_long["Buffer_Distance"].str.extract(r"(\d+)").astype(float)  #extract numbers
    
    #drop any rows where buffer distance extraction resulted in a nan value
    df_long.dropna(subset=["Buffer_Distance"], inplace=True)  #ensure clean data
    
    #convert buffer distance values to integers for proper numerical analysis
    df_long["Buffer_Distance"] = df_long["Buffer_Distance"].astype(int)
    
    #compute the mean sdi value for each unique buffer distance
    avg_mean_values = df_long.groupby("Buffer_Distance")["SDI_Value"].mean()  #aggregate by buffer distance
    
    #compute the first difference to measure the rate of change in mean sdi between successive buffers
    diff_values = np.diff(avg_mean_values).reshape(-1, 1)  #ensure 2d shape for ruptures input
    
    #initialize the ruptures pelt algorithm using the rbf model which is good for detecting gradual changes
    algo = rpt.Pelt(model="rbf").fit(diff_values)  #fit the model on the rate of change data
    
    #detect changepoints where the rate of change stabilizes using a penalty factor of 10
    changepoints = algo.predict(pen=1)  #adjust penalty value if needed for sensitivity
    
    #get the first detected changepoint if any exist
    if changepoints:
        stable_idx = changepoints[0]  #first detected changepoint index
        stable_distance = avg_mean_values.index[stable_idx]  #map index to buffer distance
        found_stabilization = True
    else:
        stable_distance = None  #no stabilization point found
        found_stabilization = False
    
    #plot the average sdi values across buffer distances to visualize stabilization
    plt.figure(figsize=(10, 6))
    plt.plot(avg_mean_values.index, avg_mean_values.values, marker="o", linestyle="-", color="b", label="Average SDI")
    if found_stabilization:
        plt.axvline(stable_distance, color='g', linestyle='--', label=f"Stabilizes at {stable_distance} feet")
    plt.xlabel("Buffer Distance (Feet)")
    plt.ylabel("Mean SDI Value")
    plt.title(f"Average SDI Across Buffer Distances - {label}")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(delta_sdi_lines_folder, f"{label}_Average_SDI.png"))
    plt.show()
    
    #plot the rate of change in sdi across buffer distances to highlight stabilization
    plt.figure(figsize=(10, 6))
    plt.plot(avg_mean_values.index[1:], np.abs(diff_values), marker="o", linestyle="-", color="r", label="Rate of Change")
    
    if found_stabilization:
        plt.axvline(stable_distance, color='g', linestyle='--', label=f"Stabilizes at {stable_distance} feet")
    
    plt.xlabel("Buffer Distance (Feet)")
    plt.ylabel("Rate of Change in Mean SDI")
    plt.title(f"Rate of Change in SDI Across Buffer Distances - {label}")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(delta_sdi_lines_folder, f"{label}_Rate_of_Change.png"))
    plt.show()

#check if user requested to delete the scratch geodatabase after processing
if delete_scratch_gdb == 'Yes':
    if arcpy.Exists(scratch_gdb_path):
        arcpy.Delete_management(scratch_gdb_path)
        print(f"Deleted: {os.path.basename(scratch_gdb_path)}")
    else:
        print(f"Geodatabase not found: {os.path.basename(scratch_gdb_path)}")